# ImageCLEF 2026 – Deepfake Detection v3
### 실행 전: 런타임 > 런타임 유형 변경 > **T4 GPU** 선택

| 섹션 | 모델 | 방식 |
|------|------|------|
| 이미지 | CLIP ViT-L/14 + dima806 ViT | KNN 앙상블 |
| 오디오 | WavLM-Large | KNN (실제 데이터 기반) |

**전체 예상 시간: 2~3시간**

## 0. 환경 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers torchaudio librosa scikit-learn pillow pandas tqdm accelerate
import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

## 1. 데이터 압축 해제

In [ ]:
import zipfile, os, glob

# ↓ 필요시 경로 수정
TEST_ZIP = '/content/drive/MyDrive/ImageCLEF2026-DeepFakeDetection-Tes.zip'
REAL_ZIP = '/content/drive/MyDrive/Real_Data_Generation_Task.zip'
TEST_DIR = '/content/test_data'
REAL_DIR = '/content/real_data'

def extract(zip_path, out_dir):
    if not os.path.exists(out_dir):
        print(f'압축 해제: {os.path.basename(zip_path)} ...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(out_dir)
        print('완료!')
    else:
        print(f'이미 해제됨: {out_dir}')

extract(TEST_ZIP, TEST_DIR)
extract(REAL_ZIP, REAL_DIR)

In [ ]:
def find_files(base, *exts):
    files = []
    for ext in exts:
        files += glob.glob(os.path.join(base, '**', f'*.{ext}'), recursive=True)
    return sorted(files)

# 테스트 데이터
test_pngs = find_files(TEST_DIR, 'png')
test_wavs = find_files(TEST_DIR, 'wav')
IMG_CSV   = glob.glob(os.path.join(TEST_DIR, '**', 'Images_Detection_submission.csv'), recursive=True)[0]
AUD_CSV   = glob.glob(os.path.join(TEST_DIR, '**', 'Audio_Detection_submission.csv'),  recursive=True)[0]
IMAGE_DIR = os.path.dirname(test_pngs[0])
AUDIO_DIR = os.path.dirname(test_wavs[0])

# 실제(Real) 참조 데이터
real_wavs = find_files(REAL_DIR, 'wav')
real_imgs = find_files(REAL_DIR, 'png', 'jpg', 'jpeg')

print(f'[테스트] 이미지: {len(test_pngs)}개 | 오디오: {len(test_wavs)}개')
print(f'[실제]   이미지: {len(real_imgs)}개 | 오디오: {len(real_wavs)}개')

## 2. 이미지 딥페이크 탐지
### 전략: CLIP KNN 점수 + dima806 모델 점수 앙상블
- **CLIP ViT-L/14**: 실제 이미지 분포 학습 → 테스트 이미지 거리 측정
- **dima806 ViT**: 딥페이크 탐지 전용 모델 예측
- **앙상블**: 두 점수 평균 → 0.5 기준 최종 분류

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('CLIP ViT-L/14 로드 중...')
clip_proc  = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
clip_model.eval()
print('완료!')

In [ ]:
def extract_clip_features(path_list, batch_size=32, desc='CLIP 특징 추출'):
    all_feats, valid_idxs = [], []
    with torch.no_grad():
        for i in tqdm(range(0, len(path_list), batch_size), desc=desc):
            batch, idx = [], []
            for j, fpath in enumerate(path_list[i:i+batch_size]):
                try:
                    batch.append(Image.open(fpath).convert('RGB'))
                    idx.append(i + j)
                except:
                    pass
            if not batch:
                continue
            inputs = clip_proc(images=batch, return_tensors='pt').to(device)
            feats  = clip_model.get_image_features(**inputs)
            # L2 정규화 (코사인 유사도용)
            feats  = feats / feats.norm(dim=-1, keepdim=True)
            feats  = feats.cpu().numpy()
            for k, vi in enumerate(idx):
                all_feats.append(feats[k])
                valid_idxs.append(vi)
    return np.array(all_feats), valid_idxs

# 실제 이미지 특징 추출
print(f'실제 이미지 {len(real_imgs)}개 처리 중...')
real_img_feats, _ = extract_clip_features(real_imgs, desc='실제 이미지')
print(f'실제 이미지 특징: {real_img_feats.shape}')

In [ ]:
import pandas as pd

img_df     = pd.read_csv(IMG_CSV)
filenames  = img_df['full_secret_name'].tolist()
test_paths = [os.path.join(IMAGE_DIR, f) for f in filenames]

print(f'테스트 이미지 {len(test_paths)}개 처리 중...')
test_img_feats, img_valid_idxs = extract_clip_features(test_paths, desc='테스트 이미지')
print(f'테스트 이미지 특징: {test_img_feats.shape}')

In [ ]:
from sklearn.neighbors import NearestNeighbors

# KNN: 이미 L2 정규화된 벡터 → inner product = cosine similarity
print('KNN 학습 중...')
knn_img = NearestNeighbors(n_neighbors=5, metric='cosine', n_jobs=-1)
knn_img.fit(real_img_feats)

# 실제 데이터 자기 거리로 임계값 보정
real_self, _ = knn_img.kneighbors(real_img_feats)
real_mu, real_sigma = real_self.mean(axis=1).mean(), real_self.mean(axis=1).std()
img_threshold = real_mu + 2.0 * real_sigma
print(f'이미지 임계값: {img_threshold:.4f}  (평균: {real_mu:.4f} ± {real_sigma:.4f})')

# 테스트 거리
test_dists, _ = knn_img.kneighbors(test_img_feats)
test_mean_dists = test_dists.mean(axis=1)

# 정규화 점수 [0,1]: 높을수록 Fake 가능성
clip_scores_raw = np.zeros(len(filenames))
for i, vi in enumerate(img_valid_idxs):
    clip_scores_raw[vi] = test_mean_dists[i] / (img_threshold + 1e-8)
clip_scores = np.clip(clip_scores_raw, 0, 2) / 2  # [0,1] 스케일

print(f'CLIP 점수 범위: {clip_scores.min():.3f} ~ {clip_scores.max():.3f}')

In [ ]:
# CLIP 모델 메모리 해제
del clip_model
torch.cuda.empty_cache()
print('CLIP 모델 해제 완료')

In [ ]:
from transformers import pipeline

print('dima806 딥페이크 탐지 모델 로드 중...')
img_pipe = pipeline(
    'image-classification',
    model='dima806/deepfake_vs_real_image_detection',
    device=0 if torch.cuda.is_available() else -1
)
print('완료!')

In [ ]:
model_scores = np.zeros(len(filenames))
BATCH_SIZE   = 32

for i in tqdm(range(0, len(filenames), BATCH_SIZE), desc='dima806 추론'):
    batch_names = filenames[i:i+BATCH_SIZE]
    batch_imgs, batch_idx = [], []

    for j, fname in enumerate(batch_names):
        try:
            batch_imgs.append(Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB'))
            batch_idx.append(i + j)
        except:
            pass

    results  = img_pipe(batch_imgs)
    res_iter = iter(results)

    for vi in batch_idx:
        r     = next(res_iter)
        top   = r[0] if isinstance(r, list) else r
        label = top['label'].lower()
        score = top['score']
        # fake이면 score 그대로, real이면 1-score
        model_scores[vi] = score if 'fake' in label else (1.0 - score)

print(f'모델 점수 범위: {model_scores.min():.3f} ~ {model_scores.max():.3f}')

In [ ]:
import shutil

# 앙상블: CLIP KNN 50% + dima806 50%
final_img_scores = 0.5 * clip_scores + 0.5 * model_scores
img_df['prediction'] = (final_img_scores >= 0.5).astype(int)

print('이미지 예측 완료!')
print(img_df['prediction'].value_counts())

img_out = '/content/Images_Detection_submission.csv'
img_df.to_csv(img_out, index=False)
shutil.copy(img_out, '/content/drive/MyDrive/Images_Detection_submission.csv')
print('이미지 CSV 저장 완료!')

# 메모리 해제
del img_pipe
torch.cuda.empty_cache()

## 3. 오디오 딥페이크 탐지
### 전략: WavLM-Large KNN (실제 데이터 기반)
- **WavLM-Large**: 음성 표현 학습 SOTA 모델 (SUPERB 벤치마크 1위)
- 실제 음성 분포 학습 → 테스트 음성 거리 측정 → 멀수록 Deepfake

In [ ]:
from transformers import AutoModel, AutoFeatureExtractor
import torchaudio

print('WavLM-Large 로드 중... (약 1~2분)')
aud_feat_extractor = AutoFeatureExtractor.from_pretrained('microsoft/wavlm-large')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-large').to(device)
wavlm.eval()
print('완료!')

In [ ]:
TARGET_SR   = 16000
MAX_SAMPLES = 16000 * 5
AUD_BATCH   = 4  # WavLM-Large는 크므로 작은 배치

def load_wav(fpath):
    wav, sr = torchaudio.load(fpath)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    wav = wav.mean(dim=0)
    return wav[:MAX_SAMPLES].numpy()

def extract_wavlm_features(path_list, desc='WavLM 특징 추출'):
    all_feats, valid_idxs = [], []
    with torch.no_grad():
        for i in tqdm(range(0, len(path_list), AUD_BATCH), desc=desc):
            batch, idx = [], []
            for j, fpath in enumerate(path_list[i:i+AUD_BATCH]):
                try:
                    batch.append(load_wav(fpath))
                    idx.append(i + j)
                except:
                    pass
            if not batch:
                continue
            inputs = aud_feat_extractor(
                batch, sampling_rate=TARGET_SR,
                return_tensors='pt', padding=True
            ).input_values.to(device)
            # (B, T, 1024) → mean pooling → (B, 1024)
            feats = wavlm(inputs).last_hidden_state.mean(dim=1).cpu().numpy()
            for k, vi in enumerate(idx):
                all_feats.append(feats[k])
                valid_idxs.append(vi)
    return np.array(all_feats), valid_idxs

# 실제 오디오 특징 추출
print(f'실제 오디오 {len(real_wavs)}개 처리 중...')
real_aud_feats, _ = extract_wavlm_features(real_wavs, desc='실제 오디오')
print(f'실제 오디오 특징: {real_aud_feats.shape}')

In [ ]:
aud_df     = pd.read_csv(AUD_CSV)
aud_names  = aud_df['full_secret_name'].tolist()
test_aud_paths = [os.path.join(AUDIO_DIR, f) for f in aud_names]

print(f'테스트 오디오 {len(test_aud_paths)}개 처리 중...')
test_aud_feats, aud_valid_idxs = extract_wavlm_features(test_aud_paths, desc='테스트 오디오')
print(f'테스트 오디오 특징: {test_aud_feats.shape}')

In [ ]:
from sklearn.preprocessing import StandardScaler

# 스케일링
scaler         = StandardScaler()
real_aud_scaled = scaler.fit_transform(real_aud_feats)
test_aud_scaled = scaler.transform(test_aud_feats)

# KNN
print('KNN 학습 중...')
knn_aud = NearestNeighbors(n_neighbors=5, metric='cosine', n_jobs=-1)
knn_aud.fit(real_aud_scaled)

# 임계값 보정
real_self_aud, _ = knn_aud.kneighbors(real_aud_scaled)
aud_mu    = real_self_aud.mean(axis=1).mean()
aud_sigma = real_self_aud.mean(axis=1).std()
aud_threshold = aud_mu + 2.0 * aud_sigma
print(f'오디오 임계값: {aud_threshold:.4f}  (평균: {aud_mu:.4f} ± {aud_sigma:.4f})')

# 테스트 분류
test_aud_dists, _ = knn_aud.kneighbors(test_aud_scaled)
test_aud_mean     = test_aud_dists.mean(axis=1)

aud_preds = np.zeros(len(aud_names), dtype=int)
for i, vi in enumerate(aud_valid_idxs):
    aud_preds[vi] = 1 if test_aud_mean[i] > aud_threshold else 0

aud_df['prediction'] = aud_preds
print('\n오디오 예측 완료!')
print(aud_df['prediction'].value_counts())

In [ ]:
aud_out = '/content/Audio_Detection_submission.csv'
aud_df.to_csv(aud_out, index=False)
shutil.copy(aud_out, '/content/drive/MyDrive/Audio_Detection_submission.csv')
print('오디오 CSV 저장 완료!')
aud_df.head(3)

## 4. 제출 파일 생성

In [ ]:
import zipfile

submit_zip = '/content/submission.zip'
with zipfile.ZipFile(submit_zip, 'w') as z:
    z.write('/content/Images_Detection_submission.csv', 'Images_Detection_submission.csv')
    z.write('/content/Audio_Detection_submission.csv',  'Audio_Detection_submission.csv')
shutil.copy(submit_zip, '/content/drive/MyDrive/submission.zip')

img_check = pd.read_csv('/content/Images_Detection_submission.csv')
aud_check = pd.read_csv('/content/Audio_Detection_submission.csv')
print('===== 최종 확인 =====')
print(f'[이미지] {len(img_check)}개 | 빈값: {img_check["prediction"].isna().sum()}')
print(img_check['prediction'].value_counts())
print(f'\n[오디오] {len(aud_check)}개 | 빈값: {aud_check["prediction"].isna().sum()}')
print(aud_check['prediction'].value_counts())
print('\n제출 파일 준비 완료: submission.zip')

In [ ]:
from google.colab import files
files.download('/content/submission.zip')